In [ ]:
import os

GEMINI_API_KEY = "GEMINI_API_KEY"
GROQ_API_KEY = "GROQ_API_KEY"
OPENAI_API_KEY = "OPENAI_API_KEY"
# Folder containing one subfolder per company (each with its BS/PL/CF CSVs).
DATA_ROOT = r"C:/MyProjects/Multiagent/Data"

# Path to your sector benchmarks Excel file.
SECTOR_SHEET_PATH = r"C:/MyProjects/Multiagent/sector_benchmarks_template.xlsx"

GEMINI_MODEL_NAME = "gemini-3-flash-preview"
GROQ_MODEL_NAME = "llama-3.3-70b-versatile"
OPENAI_MODEL_NAME = "gpt-4o-mini"

MAX_HISTORY = 5  # how many recent chat exchanges to keep as context

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY  # some library versions check this name
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [2]:
# ============================================================================
# SECTION 2 - IMPORTS
# ============================================================================
import csv
import re
from pathlib import Path
from typing import List, Literal, Optional

import pandas as pd
from pydantic import BaseModel, Field
from rapidfuzz import fuzz, process
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

DATA_ROOT = Path(DATA_ROOT)


# ============================================================================
# SECTION 3 - LLM PROVIDER SETUP (Gemini primary, Groq/OpenAI fallback)
# ============================================================================
_gemini_llm = ChatGoogleGenerativeAI(model=GEMINI_MODEL_NAME, temperature=0)
_groq_llm = ChatGroq(model=GROQ_MODEL_NAME, temperature=0)
_openai_llm = ChatOpenAI(model=OPENAI_MODEL_NAME, temperature=0)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [3]:
def build_fallback_chain(prompt, structured_output=None):
    """Build a Gemini -> Groq -> OpenAI fallback chain from a shared prompt."""
    gemini = _gemini_llm.with_structured_output(structured_output) if structured_output else _gemini_llm
    groq = _groq_llm.with_structured_output(structured_output) if structured_output else _groq_llm
    openai = _openai_llm.with_structured_output(structured_output) if structured_output else _openai_llm
    return (prompt | gemini).with_fallbacks([prompt | groq, prompt | openai])


def extract_text_content(response) -> str:
    """Gemini returns content as a list of blocks; Groq/OpenAI return a plain
    string. Handles both shapes so plain-text chains work regardless of which
    provider actually answered."""
    content = response.content
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, list):
        for block in content:
            if isinstance(block, dict) and block.get("type") == "text":
                return block["text"].strip()
            if isinstance(block, str):
                return block.strip()
        return ""
    return str(content).strip()

In [4]:
class AmbiguousCompanyError(ValueError):
    def __init__(self, user_input, candidates):
        self.candidates = candidates
        super().__init__(f"Ambiguous company '{user_input}'. Did you mean: {candidates}?")


class CompanyNotFoundError(ValueError):
    def __init__(self, user_input, available):
        self.available = available
        super().__init__(f"No folder found matching '{user_input}'.")


def _normalize_company_name(s: str) -> str:
    s = re.sub(r"[_\s]+", " ", s).strip().lower()
    s = re.sub(r"\b(ltd|limited|pvt|private)\b", "", s).strip()
    return re.sub(r"\s+", " ", s)


def list_available_companies(data_root: Path) -> list:
    return sorted([p.name for p in data_root.iterdir() if p.is_dir()])


def resolve_company_folder(data_root: Path, user_input: str, top_n=3, accept_margin=15, min_score=60) -> Path:
    """Fuzzy-match user input to a real company folder. Token-set matching
    (not character-similarity) avoids false collisions on shared words like
    'India' across many company names."""
    available = list_available_companies(data_root)
    norm_map = {_normalize_company_name(name): name for name in available}
    choices = list(norm_map.keys())
    norm_input = _normalize_company_name(user_input)

    if norm_input in norm_map:
        return data_root / norm_map[norm_input]

    results = process.extract(norm_input, choices, scorer=fuzz.token_set_ratio, limit=top_n)
    if not results or results[0][1] < min_score:
        raise CompanyNotFoundError(user_input, available)

    best_score = results[0][1]
    close_enough = [r for r in results if best_score - r[1] <= accept_margin]
    if len(close_enough) == 1:
        return data_root / norm_map[close_enough[0][0]]

    raise AmbiguousCompanyError(user_input, [norm_map[c] for c, s, i in close_enough])

In [5]:
_YEAR_PATTERN = re.compile(r"(Mar|Jun|Sep|Dec)[- ]?\d{2,4}")


def classify_statement_type(filename: str) -> str:
    name_lower = filename.lower()
    if "balance" in name_lower:
        return "BS"
    if "income" in name_lower:
        return "PL"
    if "cash" in name_lower:
        return "CF"
    return "UNKNOWN"


def discover_company_files(company_folder: Path) -> dict:
    result = {}
    for f in company_folder.glob("*.csv"):
        stype = classify_statement_type(f.name)
        if stype != "UNKNOWN":
            result[stype] = f
    return result


def clean_numeric(val) -> float:
    if val is None:
        return float("nan")
    s = str(val).strip().replace(",", "").replace('"', "")
    if s in ("", "-"):
        return float("nan")
    try:
        return float(s)
    except ValueError:
        return float("nan")

In [6]:
def load_statement_years(path: Path, n_years=5) -> dict:
    """Parse one vendor CSV into {year_label: {line_item: value}}, keeping only
    the rightmost n_years (most recent) year columns.

    Layout of these files: two title rows, three header rows (period / months /
    accounting standard), a '-' separator row, then the line items."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = [r for r in csv.reader(f)]

    # Header row = the first row containing at least 3 period labels.
    header_idx = next(
        (i for i, r in enumerate(rows) if sum(bool(_YEAR_PATTERN.match(c.strip())) for c in r) >= 3),
        None,
    )
    if header_idx is None:
        raise ValueError(f"Could not find a period-header row in {path.name}")
    header = rows[header_idx]

    year_cols = {i: c.strip() for i, c in enumerate(header) if _YEAR_PATTERN.match(c.strip())}
    keep_cols = list(year_cols)[-n_years:] # it returns list of key of the dict

    # Data starts after the '-' separator row.
    data_start = next(
        (i + 1 for i in range(header_idx + 1, len(rows)) if rows[i] and rows[i][0].strip() == "-"),
        None,
    )
    if data_start is None:
        raise ValueError(f"Could not find the '-' separator row in {path.name}")

    years = {year_cols[c]: {} for c in keep_cols} # {"Mar 24": {}, "Mar 25": {}, "Mar 26": {}}
    for row in rows[data_start:]:
        if not row:
            continue
        label = row[0].strip()
        if not label:
            continue
        for c in keep_cols:
            years[year_cols[c]][label] = clean_numeric(row[c]) if c < len(row) else float("nan") # years[year_cols[c]] access the empty dict
    return years


def load_company_statements(company_files: dict, n_years=5) -> dict:
    """{statement_type: path} -> {statement_type: {year: {line_item: value}}}"""
    return {stype: load_statement_years(path, n_years) for stype, path in company_files.items()} # 3 layered dict

In [7]:
BS_FIELDS = {
    "total_assets": ["Total assets"],
    "total_current_assets": ["Current assets and loans & advances"],
    "inventory": ["Inventories"],
    "trade_receivables": ["Trade & bills receivables"],
    "cash_and_bank": ["Cash & bank balance"],
    "total_current_liabilities": ["Current liabilities & provisions"],
    "trade_payables": ["Short term trade payables and acceptances"],
    "short_term_debt": ["Short term borrowings"],
    "long_term_debt": ["Long term borrowings excl current portion"],
    "total_equity": ["Total Capital", "Reserves and funds"],
}

PL_FIELDS = {
    "revenue": ["Net sales"],
    "cogs": ["Cost of goods sold"],
    "operating_expenses": ["Operating expenses"],
    # Excise/GST pass-through, bundled inside "Operating expenses" for
    # fuel/refining companies - needs to come back out for a meaningful
    # operating margin. Blank for most companies, which is fine.
    "indirect_taxes": ["Indirect taxes"],
    "depreciation": ["Depreciation"],
    "net_profit": ["Profit after tax"],
    # No genuine interest-expense line exists in this data source, so
    # interest coverage is not computed anywhere below.
}

In [8]:
STATEMENT_FIELDS = {"BS": BS_FIELDS, "PL": PL_FIELDS}


def standardize_year(year_values: dict, fields: dict) -> dict:
    """Collapse one year's line items into the standard field names, summing
    the components of multi-line fields (e.g. equity = capital + reserves)."""
    result = {}
    for field, components in fields.items():
        present = [year_values.get(item) for item in components]
        present = [v for v in present if v is not None and pd.notna(v)]
        result[field] = sum(present) if present else float("nan")
    return result


def standardize_statements(statements: dict) -> dict:
    """{stype: {year: {line_item: value}}} -> {stype: {year: {field: value}}}"""
    return {
        stype: {year: standardize_year(values, STATEMENT_FIELDS[stype]) for year, values in years.items()}
        for stype, years in statements.items()
        if stype in STATEMENT_FIELDS
    }

In [9]:
CATEGORY_METRICS = {
    "liquidity": ["current_ratio", "quick_ratio", "cash_ratio"],
    "profitability": ["gross_margin", "operating_margin_ebitda_based", "net_margin", "roe", "roa", "roce"],
    "efficiency": ["inventory_turnover", "receivable_turnover", "payable_turnover", "asset_turnover", "cash_conversion_cycle"],
    "leverage": ["debt_to_equity", "debt_to_asset", "debt_to_ebitda"],
}

# (display label, is_percentage) - used by print_single_result's table.
_METRIC_DISPLAY = {
    "current_ratio": ("Current ratio", False),
    "quick_ratio": ("Quick ratio", False),
    "cash_ratio": ("Cash ratio", False),
    "gross_margin": ("Gross margin", True),
    "operating_margin_ebitda_based": ("Operating margin (EBITDA)", True),
    "net_margin": ("Net margin", True),
    "roe": ("ROE", True),
    "roa": ("ROA", True),
    "roce": ("ROCE", True),
    "inventory_turnover": ("Inventory turnover", False),
    "receivable_turnover": ("Receivable turnover", False),
    "payable_turnover": ("Payable turnover", False),
    "asset_turnover": ("Asset turnover", False),
    "cash_conversion_cycle": ("Cash conversion cycle (days)", False),
    "debt_to_equity": ("Debt to equity", False),
    "debt_to_asset": ("Debt to asset", False),
    "debt_to_ebitda": ("Debt to EBITDA", False),
}


def _format_metric_value(value, is_percentage: bool) -> str:
    if value is None or (isinstance(value, float) and value != value):  # None or NaN
        return "N/A"
    return f"{value * 100:.1f}%" if is_percentage else f"{value:.2f}"


def safe_div(n, d) -> float:
    if n is None or d is None or pd.isna(n) or pd.isna(d) or d == 0:
        return float("nan")
    return n / d

In [10]:
def safe_sum(*values) -> float:
    present = [v for v in values if v is not None and pd.notna(v)]
    return sum(present) if present else float("nan")


def compute_ratios_for_year(bs: dict, pl: dict) -> dict:
    """operating_margin_ebitda_based is named that deliberately: in this data
    source, 'Operating expenses' excludes depreciation and financial-services
    costs, so Revenue - Operating Expenses lands closer to EBITDA than a
    textbook EBIT-based operating margin."""
    total_debt = safe_sum(bs.get("short_term_debt"), bs.get("long_term_debt"))

    # NaN indirect taxes means the company has no such line - treat as 0, not
    # "unknown", since most companies genuinely have no pass-through tax here.
    indirect_taxes = pl.get("indirect_taxes")
    indirect_taxes = 0 if indirect_taxes is None or pd.isna(indirect_taxes) else indirect_taxes
    adjusted_opex = (
        pl["operating_expenses"] - indirect_taxes if pd.notna(pl.get("operating_expenses")) else float("nan")
    )
    ebitda = (
        pl["revenue"] - adjusted_opex
        if pd.notna(pl.get("revenue")) and pd.notna(adjusted_opex)
        else float("nan")
    )

    r = {}
    r["current_ratio"] = safe_div(bs.get("total_current_assets"), bs.get("total_current_liabilities"))
    quick_assets = (
        bs["total_current_assets"] - bs.get("inventory", 0) if pd.notna(bs.get("total_current_assets")) else float("nan")
    )
    r["quick_ratio"] = safe_div(quick_assets, bs.get("total_current_liabilities"))
    r["cash_ratio"] = safe_div(bs.get("cash_and_bank"), bs.get("total_current_liabilities"))

    r["gross_margin"] = safe_div(pl["revenue"] - pl["cogs"], pl["revenue"]) if pd.notna(pl.get("cogs")) else float("nan")
    r["operating_margin_ebitda_based"] = safe_div(ebitda, pl.get("revenue"))
    r["net_margin"] = safe_div(pl.get("net_profit"), pl.get("revenue"))
    r["roe"] = safe_div(pl.get("net_profit"), bs.get("total_equity"))
    r["roa"] = safe_div(pl.get("net_profit"), bs.get("total_assets"))
    r["roce"] = safe_div(pl.get("net_profit"), safe_sum(bs.get("total_equity"), total_debt))

    r["inventory_turnover"] = safe_div(pl.get("cogs"), bs.get("inventory"))
    r["receivable_turnover"] = safe_div(pl.get("revenue"), bs.get("trade_receivables"))
    r["payable_turnover"] = safe_div(pl.get("cogs"), bs.get("trade_payables"))
    r["asset_turnover"] = safe_div(pl.get("revenue"), bs.get("total_assets"))
    dio = safe_div(365, r["inventory_turnover"])
    dso = safe_div(365, r["receivable_turnover"])
    dpo = safe_div(365, r["payable_turnover"])
    r["cash_conversion_cycle"] = dio + dso - dpo if all(pd.notna(x) for x in [dio, dso, dpo]) else float("nan")

    r["debt_to_equity"] = safe_div(total_debt, bs.get("total_equity"))
    r["debt_to_asset"] = safe_div(total_debt, bs.get("total_assets"))
    r["debt_to_ebitda"] = safe_div(total_debt, ebitda)
    return r

In [11]:
def compute_all_ratios(standardized: dict) -> dict:
    return {
        year: compute_ratios_for_year(standardized["BS"][year], standardized["PL"][year])
        for year in standardized["BS"]
        if year in standardized["PL"]
    }


def sanity_check_ratios(all_ratios: dict) -> list:
    warnings = []
    for year, ratios in all_ratios.items():
        for key in ["current_ratio", "quick_ratio", "cash_ratio", "inventory_turnover", "receivable_turnover",
                    "payable_turnover", "asset_turnover", "debt_to_equity", "debt_to_asset"]:
            v = ratios.get(key)
            if pd.notna(v) and v < 0:
                warnings.append(f"{year}: {key} is negative ({v}) - should never happen for this metric")
        for key in ["gross_margin", "operating_margin_ebitda_based", "net_margin"]:
            v = ratios.get(key)
            if pd.notna(v) and not (-2 < v < 1):
                warnings.append(f"{year}: {key} = {v} is outside a plausible range")
    return warnings

In [12]:
_HEADER_TO_FIELD = {
    "Sector": "sector", "Current Ratio": "current_ratio", "Quick Ratio": "quick_ratio", "Cash Ratio": "cash_ratio",
    "Gross Margin (fraction)": "gross_margin",
    "Operating Margin - EBITDA-based (fraction)": "operating_margin_ebitda_based",
    "Net Margin (fraction)": "net_margin", "ROE (fraction)": "roe", "ROA (fraction)": "roa",
    "ROCE (fraction)": "roce", "Inventory Turnover": "inventory_turnover",
    "Receivable Turnover": "receivable_turnover", "Payable Turnover": "payable_turnover",
    "Asset Turnover": "asset_turnover", "Cash Conversion Cycle (days)": "cash_conversion_cycle",
    "Debt to Equity": "debt_to_equity", "Debt to Asset": "debt_to_asset", "Debt to EBITDA": "debt_to_ebitda",
    "P/E Ratio (optional - valuation)": "pe_ratio", "P/B Ratio (optional - valuation)": "pb_ratio",
    "EV/EBITDA (optional - valuation)": "ev_ebitda", "EV/Sales (optional - valuation)": "ev_sales",
    "Dividend Yield (optional - valuation, fraction)": "dividend_yield",
    "PEG Ratio (optional - valuation)": "peg_ratio", "Source / date (for auditability)": "source",
}
_SECTOR_STOPWORDS = {"and", "&", "of", "the", "for", "in", "on", "a", "an"}


def load_sector_benchmarks(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="Sector Benchmarks", header=0)
    return df.rename(columns=lambda c: _HEADER_TO_FIELD.get(str(c).strip(), str(c).strip()))


def _normalize_sector(s: str) -> str:
    s = re.sub(r"[^a-z0-9\s]", " ", s.lower().strip())
    return " ".join(w for w in s.split() if w not in _SECTOR_STOPWORDS)


def resolve_sector_to_benchmark_row(candidate_sector: str, available_sectors: list, min_score=65):
    """Strips connector words before matching - without this, short sector
    names sharing a word like 'and' can produce false matches (e.g. 'Cables
    and Wires' incorrectly matching 'Garments and Apparels')."""
    if not available_sectors:
        return None
    norm_map = {_normalize_sector(s): s for s in available_sectors}
    result = process.extractOne(_normalize_sector(candidate_sector), list(norm_map.keys()), scorer=fuzz.token_set_ratio)
    if result is None or result[1] < min_score:
        return None
    return norm_map[result[0]]


def get_sector_benchmark(df: pd.DataFrame, sector: str, metrics: list) -> dict:
    match = df[df["sector"] == sector]
    if match.empty:
        return {m: None for m in metrics}
    row = match.iloc[0]
    return {m: (None if pd.isna(row.get(m)) else row.get(m)) for m in metrics}

In [13]:
_sector_prompt = ChatPromptTemplate.from_template("""
You are a financial analyst.
Classify the following company into the MOST RELEVANT sector
from the provided sector list.
Company:
{company_name}
Available sectors:
{sectors}
Rules:
1. Select exactly ONE sector.
2. The selected sector MUST come from the provided list.
3. Do not create a new sector.
4. Return only the selected sector name.
""")


def classify_sector(company_name: str, available_sectors: list) -> str:
    chain = build_fallback_chain(_sector_prompt, structured_output=None)
    response = chain.invoke({"company_name": company_name, "sectors": "\n".join(available_sectors)})
    return extract_text_content(response)

In [14]:
class CategoryInterpretation(BaseModel):
    headline: str = Field(description="One-line verdict on this category")
    narrative: str = Field(description="A short paragraph interpreting the trend across the 5 years")
    notable_points: List[str] = Field(description="2-4 specific, concrete observations")


_INTERPRETATION_SYSTEM_PROMPT = """You are a financial analyst writing interpretation \
for one category of ratios, for a company in the sector given to you.

Rules:
- Interpret the trend across the 5 years given - do not recalculate or invent numbers.
- Be specific: reference actual values and direction of change.
- If this is the leverage category: interest coverage ratio is NOT available - do not \
mention or reason about it.
- Keep the narrative to one short paragraph.
- CRITICAL: "lower than expected" is NOT the same as "negative". A margin or ratio \
can decline while remaining positive. Only call a value negative if it's actually \
below zero in the data given to you."""

_interpretation_prompt = ChatPromptTemplate.from_messages([
    ("system", _INTERPRETATION_SYSTEM_PROMPT),
    ("human", "Category: {category}\n\nSector: {sector}\n\n5-year data:\n{category_data}"),
])

In [15]:
def _get_category_series(all_ratios: dict, metrics: list) -> dict:
    return {m: {year: all_ratios[year].get(m) for year in all_ratios} for m in metrics}


def _format_category_data(category_series: dict) -> str:
    lines = []
    for metric, year_values in category_series.items():
        vals = ", ".join(f"{y}: {round(v, 3) if v == v else 'N/A'}" for y, v in year_values.items())
        lines.append(f"{metric} -> {vals}")
    return "\n".join(lines)


def interpret_category(category: str, metrics: list, all_ratios: dict, sector: str) -> dict:
    chain = build_fallback_chain(_interpretation_prompt, structured_output=CategoryInterpretation)
    series = _get_category_series(all_ratios, metrics)
    result = chain.invoke({"category": category, "sector": sector, "category_data": _format_category_data(series)})
    return result.model_dump()

In [16]:
class SynthesisResult(BaseModel):
    overall_headline: str
    company_vs_sector: str = Field(description="Explicitly note where benchmarks weren't found rather than skipping silently")
    trajectory: str
    outlook_notes: str = Field(description="Analysis, not a recommendation or prediction")
    key_strengths: List[str]
    key_watch_items: List[str]


_SYNTHESIS_SYSTEM_PROMPT = """You synthesize a full financial analysis, combining 5-year \
ratio trends, category interpretations, and sector benchmarks (where available).

Rules:
- This is financial analysis, not investment advice. Never recommend buying, selling, \
or holding, and never give a price target.
- Where sector benchmark data wasn't found, say so explicitly.
- Ground every claim in the data you were given.
- CRITICAL: "lower than the sector" and "negative" are NOT the same thing. Only call a \
value negative if it is actually below zero - check the sign before using that word."""

_synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", _SYNTHESIS_SYSTEM_PROMPT),
    ("human", "Company: {company}\nSector: {sector}\n\nRatios (5yr):\n{ratios}\n\n"
              "Category interpretations:\n{interpretations}\n\nSector benchmarks:\n{benchmarks}"),
])

In [17]:
def synthesize(company: str, sector: str, all_ratios: dict, interpretations: dict, sector_benchmarks: dict) -> dict:
    chain = build_fallback_chain(_synthesis_prompt, structured_output=SynthesisResult)
    result = chain.invoke({
        "company": company, "sector": sector, "ratios": all_ratios,
        "interpretations": interpretations, "benchmarks": sector_benchmarks,
    })
    return result.model_dump()

In [18]:
def run_single_company_analysis(company_input: str) -> dict:
    """Resolve -> load -> standardize -> compute ratios -> classify sector ->
    interpret each category -> synthesize. The data layer is fully
    deterministic; each LLM step degrades gracefully to a clear 'unavailable'
    placeholder rather than crashing the run."""
    folder = resolve_company_folder(DATA_ROOT, company_input)
    files = discover_company_files(folder)
    statements = load_company_statements(files, n_years=5)
    standardized = standardize_statements(statements)

    all_ratios = compute_all_ratios(standardized)
    company_name = folder.name.replace("_", " ")

    for warning in sanity_check_ratios(all_ratios):
        print(f"  [SANITY CHECK] {warning}")

    all_metrics = [m for metrics in CATEGORY_METRICS.values() for m in metrics]
    try:
        benchmark_df = load_sector_benchmarks(SECTOR_SHEET_PATH)
        available_sectors = benchmark_df["sector"].dropna().tolist()
        raw_sector = classify_sector(company_name, available_sectors)
        print(f"  [Sector classification returned: '{raw_sector}']")
        resolved_sector = resolve_sector_to_benchmark_row(raw_sector, available_sectors)
        if resolved_sector is None:
            print(f"  [WARNING: '{raw_sector}' did not match any sector in your sheet closely "
                  f"enough - proceeding without a sector benchmark comparison]")
        benchmarks = (
            get_sector_benchmark(benchmark_df, resolved_sector, all_metrics)
            if resolved_sector else {m: None for m in all_metrics}
        )
    except Exception as e:
        print(f"  [Sector step failed: {type(e).__name__}: {e} - proceeding without sector benchmarks]")
        raw_sector = None
        resolved_sector = None
        benchmarks = {m: None for m in all_metrics}

    sector_for_context = resolved_sector or raw_sector or "unknown sector"

    interpretations = {}
    for category, metrics in CATEGORY_METRICS.items():
        try:
            interpretations[category] = interpret_category(category, metrics, all_ratios, sector_for_context)
        except Exception as e:
            interpretations[category] = {
                "headline": "Interpretation unavailable",
                "narrative": f"Could not generate interpretation ({type(e).__name__}).",
                "notable_points": [],
            }

    try:
        synthesis_result = synthesize(company_name, sector_for_context, all_ratios, interpretations, benchmarks)
    except Exception as e:
        synthesis_result = {
            "overall_headline": "Synthesis unavailable",
            "company_vs_sector": f"Could not generate synthesis ({type(e).__name__}).",
            "trajectory": "", "outlook_notes": "", "key_strengths": [], "key_watch_items": [],
        }

    return {
        "company": company_name, "sector": sector_for_context, "all_ratios": all_ratios,
        "benchmarks": benchmarks, "interpretations": interpretations, "synthesis": synthesis_result,
    }

In [19]:
def print_single_result(result: dict) -> None:
    print(f"\n=== {result['company']} vs {result['sector']} sector ===")
    latest_year = list(result["all_ratios"].keys())[-1]
    company_values = result["all_ratios"][latest_year]
    sector_values = result["benchmarks"]

    print(f"(latest year: {latest_year})")
    for category, metrics in CATEGORY_METRICS.items():
        print(f"\n{category.upper()}")
        header = f"  {'Attribute':<28} {'Company':>12} {'Sector':>12}"
        print(header)
        print("  " + "-" * (len(header) - 2))
        for metric_key in metrics:
            label, is_pct = _METRIC_DISPLAY.get(metric_key, (metric_key, False))
            company_val = _format_metric_value(company_values.get(metric_key), is_pct)
            sector_val = _format_metric_value(sector_values.get(metric_key), is_pct)
            print(f"  {label:<28} {company_val:>12} {sector_val:>12}")

    print(f"\n{result['synthesis']['overall_headline']}")
    print(result["synthesis"]["company_vs_sector"])

In [20]:
class ComparisonResult(BaseModel):
    headline: str
    comparison: str = Field(description="The comparative narrative, grounded in the actual numbers given")
    company_a_strengths: List[str]
    company_b_strengths: List[str]


_COMPARISON_SYSTEM_PROMPT = """You are a financial analyst comparing two companies using \
their 5-year ratio data. Be specific, cite actual numbers you were given, and never \
invent a figure not present in the data. This is analysis, not investment advice."""

_comparison_prompt = ChatPromptTemplate.from_messages([
    ("system", _COMPARISON_SYSTEM_PROMPT),
    ("human", "Company A: {company_a}\nRatios A (latest year): {ratios_a}\n\n"
              "Company B: {company_b}\nRatios B (latest year): {ratios_b}\n\n"
              "Write a comparative analysis."),
])

In [21]:
def run_comparison(result_a: dict, result_b: dict) -> dict:
    chain = build_fallback_chain(_comparison_prompt, structured_output=ComparisonResult)
    latest_a = list(result_a["all_ratios"].keys())[-1]
    latest_b = list(result_b["all_ratios"].keys())[-1]
    result = chain.invoke({
        "company_a": result_a["company"], "ratios_a": result_a["all_ratios"][latest_a],
        "company_b": result_b["company"], "ratios_b": result_b["all_ratios"][latest_b],
    })
    return result.model_dump()


def print_comparison(company_a: str, company_b: str, comparison: dict) -> None:
    print(f"\n=== {company_a} vs {company_b} ===")
    print(comparison["headline"])
    print(f"\n{comparison['comparison']}")
    print(f"\n{company_a} strengths:")
    for s in comparison["company_a_strengths"]:
        print(f"  - {s}")
    print(f"\n{company_b} strengths:")
    for s in comparison["company_b_strengths"]:
        print(f"  - {s}")

In [22]:
class QueryIntent(BaseModel):
    intent: Literal["new_analysis", "comparison", "app_question", "off_topic", "clarify"]
    company_name: Optional[str] = Field(default=None, description="Company to analyze, or the NEW company in a comparison")
    compare_with: Optional[str] = Field(default=None, description="'previous' for a vague reference, or a named earlier company")


_INTENT_SYSTEM_PROMPT = """You are the router for a financial analysis chat tool covering \
NIFTY 50 companies.

Given the user's latest message and recent conversation history (companies already \
analyzed in this session), classify their intent:

- "new_analysis": wants a fresh single-company analysis. Extract company_name.
- "comparison": wants to compare companies. If they refer to "previous"/"that company"/"it" \
without naming it, set compare_with="previous". If they name both a new company AND \
an earlier one, company_name = the new one, compare_with = the earlier one.
- "app_question": asking what this tool does.
- "off_topic": anything else.
- "clarify": clearly a company/comparison request but you can't tell which company.

Conversation history (most recent last):
{history}""" # placeholder

_intent_prompt = ChatPromptTemplate.from_messages([("system", _INTENT_SYSTEM_PROMPT), ("human", "{user_input}")]) # placeholder

In [23]:
def format_history(history: list) -> str:
    if not history:
        return "(no prior conversation)"
    return "\n".join(f'User said: "{t["user_input"]}"\n  -> analyzed: {t["company"]}' for t in history)


def find_in_history(name: str, history: list) -> Optional[dict]:
    """Reuses an already-computed result if the company was analyzed earlier
    this session, instead of re-running the whole pipeline unnecessarily."""
    for turn in reversed(history):
        if name.lower() in turn["company"].lower() or turn["company"].lower() in name.lower():
            return turn["result"]
    return None

In [24]:
def chat() -> None:
    """Starts the interactive loop. In Jupyter, run this function and type into
    the prompt that appears below the cell."""
    print("Financial Analysis Chat - type a company name, or 'exit' to quit.")
    print("Try: 'ashok leyland', then later 'compare it with bajaj auto'.\n")

    history: list = []
    intent_chain = build_fallback_chain(_intent_prompt, structured_output=QueryIntent)

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye.")
            break

        if user_input.lower() in ("exit", "quit"):
            print("Goodbye.")
            break
        if not user_input:
            continue

        recent_history = history[-MAX_HISTORY:]
        try:
            intent_result = intent_chain.invoke({"user_input": user_input, "history": format_history(recent_history)})
        except Exception as e:
            print(f"\nSomething went wrong understanding that ({type(e).__name__}: {e}). Try rephrasing.")
            continue

        if intent_result.intent == "app_question":
            print("\nI analyze NIFTY 50 companies' 5-year financials. Give me a name, "
                  "or ask me to compare two once you've analyzed at least one.")
            continue
        if intent_result.intent == "off_topic":
            print("\nI'm built for NIFTY 50 financial analysis - try a company name.")
            continue
        if intent_result.intent == "clarify":
            print("\nWhich company did you mean?")
            continue

        if intent_result.intent == "new_analysis":
            if not intent_result.company_name:
                print("\nWhich company?")
                continue
            try:
                result = run_single_company_analysis(intent_result.company_name)
            except (AmbiguousCompanyError, CompanyNotFoundError) as e:
                print(f"\n{e}")
                continue
            except Exception as e:
                print(f"\nSomething went wrong analyzing that ({type(e).__name__}: {e}).")
                continue
            print_single_result(result)
            history.append({"user_input": user_input, "company": result["company"], "result": result})

        elif intent_result.intent == "comparison":
            prior_result = None
            if intent_result.compare_with and intent_result.compare_with.lower() != "previous":
                prior_result = find_in_history(intent_result.compare_with, recent_history)
                if prior_result is None:
                    try:
                        prior_result = run_single_company_analysis(intent_result.compare_with)
                    except (AmbiguousCompanyError, CompanyNotFoundError) as e:
                        print(f"\n{e}")
                        continue
            elif recent_history:
                prior_result = recent_history[-1]["result"]
            else:
                print("\nI don't have a previous company to compare against yet - analyze one first.")
                continue

            if intent_result.company_name:
                new_result = find_in_history(intent_result.company_name, recent_history)
                if new_result is None:
                    try:
                        new_result = run_single_company_analysis(intent_result.company_name)
                    except (AmbiguousCompanyError, CompanyNotFoundError) as e:
                        print(f"\n{e}")
                        continue
            else:
                print("\nWhich company should I compare against?")
                continue

            try:
                comparison = run_comparison(new_result, prior_result)
            except Exception as e:
                print(f"\nSomething went wrong comparing those ({type(e).__name__}: {e}).")
                continue

            print_comparison(new_result["company"], prior_result["company"], comparison)
            history.append({"user_input": user_input, "company": new_result["company"], "result": new_result})

In [25]:
chat()

Financial Analysis Chat - type a company name, or 'exit' to quit.
Try: 'ashok leyland', then later 'compare it with bajaj auto'.




You:  financial analysis for ITC


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


  [Sector classification returned: 'Diversified FMCG']

=== ITC vs Diversified FMCG sector ===
(latest year: Mar 2026)

LIQUIDITY
  Attribute                         Company       Sector
  ------------------------------------------------------
  Current ratio                        3.02         2.05
  Quick ratio                          1.85         1.47
  Cash ratio                           0.14         1.00

PROFITABILITY
  Attribute                         Company       Sector
  ------------------------------------------------------
  Gross margin                        45.8%        50.0%
  Operating margin (EBITDA)           31.6%        26.5%
  Net margin                          28.2%        19.5%
  ROE                                 29.0%        29.0%
  ROA                                 22.6%        19.0%
  ROCE                                28.1%        35.0%

EFFICIENCY
  Attribute                         Company       Sector
  -------------------------------------------


You:  exit


Goodbye.
